# FREIA McStas detector data

This notebook visualizes the final detector's image, arrival-time distribution, and wavelength spectrum for a FREIA simulation. Wavelengths are reconstructed using the WFM chopper settings.


In [ ]:
%matplotlib widget
import plopp as pp
import scipp as sc

from ess import freia
from ess.freia import data
from ess.reduce.nexus.types import DiskChoppers, Filename, RawDetector
from ess.reduce.unwrap import (
    ChopperFrameSequence,
    DistanceResolution,
    PulsePeriod,
    TimeResolution,
    WavelengthDetector,
)
from ess.reduce.unwrap.types import KeepEventTimeOffset
from ess.reflectometry.types import SampleRun


## Select a run

Select the sample simulation and set the resolution used to reconstruct wavelengths.


In [ ]:
freia_mcstas = freia.FreiaMcStasWorkflow(wavelength_from='analytical')
freia_mcstas[Filename[SampleRun]] = data.freia_mcstas_sample_run()
freia_mcstas[KeepEventTimeOffset] = True
freia_mcstas[TimeResolution] = sc.scalar(20.0, unit='us')
freia_mcstas[DistanceResolution] = sc.scalar(0.1, unit='m')


## Inspect the WFM chopper cascade

This example uses a fixed **WFM** configuration with three bandwidth disks and five WFM disks with seven openings each.

For another configuration, assign its chopper dictionary to `freia_mcstas[DiskChoppers[SampleRun]]`.

The analytical model approximates the beam by a central ray. It projects chopper positions onto the global z axis and estimates detector flight paths as source-to-sample plus sample-to-pixel distance; it does not trace the curved guide or model the finite beam footprint on each disk.


In [ ]:
choppers = freia_mcstas.compute(DiskChoppers[SampleRun])
sc.DataGroup(choppers)


In [ ]:
frames = freia_mcstas.compute(ChopperFrameSequence[SampleRun])
frames.draw()


## Load and unwrap the detector events

Compute wavelengths from the event arrival times and chopper transmission bands.


In [ ]:
results = freia_mcstas.compute((RawDetector[SampleRun], WavelengthDetector[SampleRun]))
raw = results[RawDetector[SampleRun]]
unwrapped = results[WavelengthDetector[SampleRun]]
raw


## Detector image and arrival times

The image uses `longitude` and `height` in the detector's local frame. Intensities are sums of event weights.


In [ ]:
detector_image = raw.hist(longitude=120, height=80, dim=raw.dims)
pp.plot(detector_image, norm='log', title='FREIA final detector')


In [ ]:
arrival_times = raw.hist(
    event_time_offset=sc.linspace('event_time_offset', 0.0, freia_mcstas.compute(PulsePeriod).to(unit='s').value, 501, unit='s'),
    dim=raw.dims,
)
arrival_times.coords['event_time_offset'] = arrival_times.coords['event_time_offset'].to(unit='ms')
pp.plot(arrival_times, title='Arrival time within the source period')


## Wavelengths from analytical frame unwrapping

The chopper cascade and arrival times determine the wavelengths. Events outside the modeled transmission bands or above the workflow's relative wavelength uncertainty threshold have NaN wavelengths and do not contribute to the wavelength histograms. Inspect the assigned-event count when assessing the result.


In [ ]:
event_wavelengths = unwrapped.bins.constituents['data'].coords['wavelength']
valid = sc.isfinite(event_wavelengths)
print(f'{sc.sum(valid).value:,} / {valid.size:,} events have an assigned wavelength')

wavelength_bins = sc.linspace('wavelength', 1.0, 12.0, 441, unit='angstrom')
spectrum = unwrapped.hist(wavelength=wavelength_bins, dim=unwrapped.dims)
pp.plot(spectrum, title='FREIA wavelength spectrum')


In [ ]:
wavelength_image = unwrapped.hist(
    wavelength=wavelength_bins, longitude=120, dim=unwrapped.dims
)
pp.plot(wavelength_image, norm='log', title='Wavelength across the detector')
